# Notebook 3 – Feature Engineering

Objective

• Create customer behavioral features from transaction history.

• Generate RFM, trend, lifetime, and purchase gap features.

• Transform raw transactional data into model-ready variables.

Goal:
Convert each customer snapshot into numerical behavioral features.

Feature Groups:

1. Recent Behavior
   - Recency
   - Frequency_180d
   - Monetary_180d
   - Average Order Value

2. Trend Features
   - Spend Trend
   - Frequency Trend
   - AOV Trend

3. Purchase Pattern Features
   - Average Purchase Gap
   - Purchase Variability

4. Lifetime Features
   - Customer Age
   - Lifetime Spend
   - Lifetime Orders

Output:
One ML-ready row per customer snapshot.

In [ ]:
# ====================================================
# MOUNT GOOGLE DRIVE
# ====================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ====================================================
# IMPORT LIBRARIES
# ====================================================

import pandas as pd
import numpy as np

# ====================================================
# LOAD INPUT DATASETS
# ====================================================

retail = pd.read_pickle(
    "/content/drive/MyDrive/Project files/retail_clean.pkl"
)

snapshot_df = pd.read_pickle(
    "/content/drive/MyDrive/Project files/snapshot_df.pkl"
)

print(retail.shape)
print(snapshot_df.shape)

(805549, 10)
(46098, 2)


In [ ]:
#Step 1 — Create Invoice-Level Dataset
# ====================================================
# CREATE INVOICE-LEVEL DATASET
# ====================================================

# One invoice can contain many products.
# We want one row per invoice (order).

invoice_level = (
    retail
    .groupby(
        ["Customer ID", "Invoice", "InvoiceDate"],
        as_index=False
    )
    .agg(
        Revenue=("Revenue", "sum")
    )
)

# Check shape
print("Invoice Level Shape:")
print(invoice_level.shape)

# Display first rows
invoice_level.head()

Invoice Level Shape:
(37033, 4)


,Customer ID,Invoice,InvoiceDate,Revenue
0,12346,491725,2009-12-14 08:34:00,45.0
1,12346,491742,2009-12-14 11:00:00,22.5
2,12346,491744,2009-12-14 11:02:00,22.5
3,12346,492718,2009-12-18 10:47:00,22.5
4,12346,492722,2009-12-18 10:55:00,1.0


In [ ]:
#Step 2 — Check Customer Order Statistics

# Number of orders per customer

customer_orders = (
    invoice_level
    .groupby("Customer ID")["Invoice"]
    .nunique()
)

print(customer_orders.describe())

count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: Invoice, dtype: float64


In [ ]:
#Step 3 — Rebuild the Merge
# ====================================================
# MERGE SNAPSHOTS WITH ORDERS
# ====================================================

merged_df = snapshot_df.merge(
    invoice_level,
    on="Customer ID",
    how="inner"
)

print("Merged Shape:")
print(merged_df.shape)

merged_df.head()

Merged Shape:
(415975, 5)


,Customer ID,snapshot_date,Invoice,InvoiceDate,Revenue
0,12362,2010-05-31,489447,2009-12-01 10:10:00,130.00
1,12362,2010-05-31,544203,2011-02-17 10:30:00,479.10
2,12362,2010-05-31,551346,2011-04-28 09:12:00,495.24
3,12362,2010-05-31,559295,2011-07-07 12:32:00,303.76
4,12362,2010-05-31,563037,2011-08-11 15:02:00,469.25


In [ ]:
# Step 4 — Keep Only Orders Before Snapshot
#Keep only orders that happened
# before the snapshot date

history_df = merged_df[
    merged_df["InvoiceDate"]
    < merged_df["snapshot_date"]
].copy()

print("History Shape:")
print(history_df.shape)

History Shape:
(268060, 5)


In [ ]:
#Step 5 — Keep Only Last 180 Days
# Keep only orders inside the
# previous 180-day observation window

history_180d = history_df[
    history_df["InvoiceDate"]
    >= (
        history_df["snapshot_date"]
        - pd.Timedelta(days=180)
    )
].copy()

print("180-Day History Shape:")
print(history_180d.shape)

history_180d.head()

180-Day History Shape:
(96977, 5)


,Customer ID,snapshot_date,Invoice,InvoiceDate,Revenue
12,12490,2010-05-31,496325,2010-01-31 13:05:00,320.26
13,12490,2010-05-31,496916,2010-02-04 15:33:00,51.00
14,12490,2010-05-31,497377,2010-02-09 11:37:00,534.26
15,12490,2010-05-31,500121,2010-03-04 14:05:00,202.35
16,12490,2010-05-31,500582,2010-03-09 09:46:00,400.20


### Feature Group 1: Trend Features

#Part A — Feature 1: Recency
Business Question
How many days ago did the customer last buy?

In [ ]:
# ====================================================
# FEATURE 1 : RECENCY
# ====================================================

# Find most recent order before each snapshot

last_purchase = (
    history_180d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["InvoiceDate"]
    .max()
    .reset_index()
)

# Rename column

last_purchase.rename(
    columns={
        "InvoiceDate": "last_purchase_date"
    },
    inplace=True
)

# Calculate days since last purchase

last_purchase["recency_days"] = (
    last_purchase["snapshot_date"]
    - last_purchase["last_purchase_date"]
).dt.days

# Merge into snapshot dataset

snapshot_features = snapshot_df.merge(
    last_purchase[
        [
            "Customer ID",
            "snapshot_date",
            "recency_days"
        ]
    ],
    on=[
        "Customer ID",
        "snapshot_date"
    ],
    how="left"
)

# Check result

snapshot_features.head()

,Customer ID,snapshot_date,recency_days
0,12362,2010-05-31,NaN
1,12490,2010-05-31,23.0
2,12533,2010-05-31,69.0
3,12636,2010-05-31,NaN
4,12682,2010-05-31,24.0


In [ ]:
#Audit Recency
snapshot_features["recency_days"].describe()

,recency_days
count,27154.000000
mean,61.797746
std,49.879508
min,0.000000
25%,19.000000
50%,48.000000
75%,98.000000
max,179.000000


#Part B — Feature 2: Frequency_180d
Business Question
How many orders did the customer place
in the last 180 days?

In [ ]:
# ====================================================
# FEATURE 2 : FREQUENCY_180D
# ====================================================

frequency = (
    history_180d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Invoice"]
    .nunique()
    .reset_index()
)

frequency.rename(
    columns={
        "Invoice": "frequency_180d"
    },
    inplace=True
)

snapshot_features = snapshot_features.merge(
    frequency,
    on=[
        "Customer ID",
        "snapshot_date"
    ],
    how="left"
)

snapshot_features.head()

,Customer ID,snapshot_date,recency_days,frequency_180d
0,12362,2010-05-31,NaN,NaN
1,12490,2010-05-31,23.0,6.0
2,12533,2010-05-31,69.0,1.0
3,12636,2010-05-31,NaN,NaN
4,12682,2010-05-31,24.0,9.0


#Part C — Feature 3: Monetary_180d
Business Question
How much revenue did the customer generate
during the last 180 days?

In [ ]:
# ====================================================
# FEATURE 3 : MONETARY_180D
# ====================================================

monetary = (
    history_180d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Revenue"]
    .sum()
    .reset_index()
)

monetary.rename(
    columns={
        "Revenue": "monetary_180d"
    },
    inplace=True
)

snapshot_features = snapshot_features.merge(
    monetary,
    on=[
        "Customer ID",
        "snapshot_date"
    ],
    how="left"
)

snapshot_features.head()

,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d
0,12362,2010-05-31,NaN,NaN,NaN
1,12490,2010-05-31,23.0,6.0,1943.23
2,12533,2010-05-31,69.0,1.0,437.97
3,12636,2010-05-31,NaN,NaN,NaN
4,12682,2010-05-31,24.0,9.0,5325.20


# Part D — Feature 4: Average Order Value
Business Question
How much does the customer spend per order?

In [ ]:
# ====================================================
# FEATURE 4 : AOV
# ====================================================

snapshot_features["aov_180d"] = (
    snapshot_features["monetary_180d"]
    /
    snapshot_features["frequency_180d"]
)

In [ ]:
snapshot_features[
    [
        "Customer ID",
        "snapshot_date",
        "recency_days",
        "frequency_180d",
        "monetary_180d",
        "aov_180d"
    ]
].head()

,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d
0,12362,2010-05-31,NaN,NaN,NaN,NaN
1,12490,2010-05-31,23.0,6.0,1943.23,323.871667
2,12533,2010-05-31,69.0,1.0,437.97,437.970000
3,12636,2010-05-31,NaN,NaN,NaN,NaN
4,12682,2010-05-31,24.0,9.0,5325.20,591.688889


In [ ]:
snapshot_features.describe()

,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d
count,46098,27154.000000,27154.000000,27154.000000,27154.000000
mean,2011-03-17 05:26:20.476376320,61.797746,3.563342,1783.291595,433.061625
min,2010-05-31 00:00:00,0.000000,1.000000,5.100000,4.950000
25%,2010-12-31 00:00:00,19.000000,1.000000,321.865000,198.960000
50%,2011-03-31 00:00:00,48.000000,2.000000,697.330000,313.350000
75%,2011-06-30 00:00:00,98.000000,4.000000,1528.370000,472.884375
max,2011-08-31 00:00:00,179.000000,126.000000,207082.990000,77183.600000
std,NaN,49.879508,5.415922,6116.763794,1239.349383


#This is actually a very useful result because it exposed something important.

### Snapshot Coverage Analysis

- Total snapshots generated: **46,098**
- Snapshots with available Recency, Frequency, and Monetary (RFM) values: **27,154**
- Difference: **18,944 snapshots**

#### Reason

- Snapshots were generated using the rule:

  **Customer First Purchase Date + 180 Days**

- The rule did **not** require the customer to have a purchase within the previous 180 days.
- Therefore, some customers received snapshots despite having no recent transaction activity.

#### Example

- Customer made a purchase in **January 2010**.
- No further purchases were made.
- A snapshot was still generated in **July 2011**.
- Since there were no transactions in the previous 180 days:
  - Frequency_180d = 0
  - Monetary_180d = 0

#### Interpretation

- These 18,944 snapshots are **not data errors**.
- They represent customers who were inactive during the observation window.
- Such customers are likely to be at a higher risk of value decline or churn.
- Therefore, these snapshots contain useful information and can help the model learn patterns associated with declining customer activity.


In [ ]:
snapshot_features.isnull().sum()

,0
Customer ID,0
snapshot_date,0
recency_days,18944
frequency_180d,18944
monetary_180d,18944
aov_180d,18944


In [ ]:
# Customers with no activity
# in previous 180 days

snapshot_features["recency_days"] = (
    snapshot_features["recency_days"]
    .fillna(180)
)

snapshot_features["frequency_180d"] = (
    snapshot_features["frequency_180d"]
    .fillna(0)
)

snapshot_features["monetary_180d"] = (
    snapshot_features["monetary_180d"]
    .fillna(0)
)

snapshot_features["aov_180d"] = (
    snapshot_features["aov_180d"]
    .fillna(0)
)

### Handling Customers with No Recent Purchases

* Some customer snapshots contain **no purchases within the previous 180-day observation window**.
* These customers are retained because their inactivity represents meaningful customer behavior and may indicate a higher risk of value decline.

#### Handling Strategy

* Recency → 180
* Frequency → 0
* Monetary → 0
* AOV → 0

### Feature Quality Check

#### Recency

* Median = 48 days
* 75th Percentile = 98 days
* Maximum = 179 days
* Values are within the expected 180-day observation window.

#### Frequency

* Median = 2 orders
* 75th Percentile = 4 orders
* Values are consistent with the transaction audit.

#### Monetary

* Median ≈ £697
* Indicates a reasonable level of customer spending.

#### AOV

* Median ≈ £313
* Falls within an expected range.

### Conclusion

* Feature distributions appear realistic and consistent with customer purchasing behavior.
* No significant data quality issues were identified during feature validation.

In [ ]:
snapshot_features[
[
    "recency_days",
    "frequency_180d",
    "monetary_180d",
    "aov_180d"
]
].describe()

,recency_days,frequency_180d,monetary_180d,aov_180d
count,46098.000000,46098.000000,46098.000000,46098.000000
mean,110.373031,2.098985,1050.446873,255.094697
std,69.625682,4.511274,4775.841632,974.760429
min,0.000000,0.000000,0.000000,0.000000
25%,38.000000,0.000000,0.000000,0.000000
50%,125.000000,1.000000,225.200000,153.510000
75%,180.000000,3.000000,871.567500,347.092500
max,180.000000,126.000000,207082.990000,77183.600000




### Final RFM Feature Audit
These numbers look much more realistic because we've handled the inactive customers properly.
#### Recency

* Median = 125 days
* 75th Percentile = 180 days
* Maximum = 180 days

**Interpretation:**

* Around 50% of customer snapshots have not made a purchase for at least 125 days.
* Approximately 25% of snapshots had no purchases within the previous 180-day observation window, resulting in a Recency value of 180.
* Customer inactivity is an important indicator and may serve as a strong signal of future value decline.

#### Frequency_180d

* Median = 1 order
* 75th Percentile = 3 orders
* Maximum = 126 orders

**Interpretation:**

* Most customers place between 1 and 3 orders within a 180-day period.
* A small number of customers place significantly more orders, likely representing high-frequency or business customers.

#### Monetary_180d

* Median = £225
* 75th Percentile = £872

**Interpretation:**

* Customer spending shows a typical retail distribution.
* Most customers contribute relatively small amounts of revenue, while a smaller group contributes substantially higher spending.

#### AOV_180d

* Median = £153
* 75th Percentile = £347

**Interpretation:**

* Average Order Value distribution appears reasonable and aligns with expected customer purchasing behavior.

### What Has Been Built?

* Created customer-level RFM features using a 180-day observation window.
* Generated the first machine learning features for each customer snapshot.
* Captured both active and inactive customer behavior, enabling the model to learn patterns associated with future value decline.

**Example Customer Snapshots**

* Customer 12345 → Snapshot Date: 2011-07-31 → Recency: 15 → Frequency: 3 → Monetary: 900 → AOV: 300

* Customer 12346 → Snapshot Date: 2011-07-31 → Recency: 180 → Frequency: 0 → Monetary: 0 → AOV: 0

* The resulting dataset is now ready for the next stage of feature engineering and target creation.



In [ ]:
# Save current feature dataset

snapshot_features.to_csv(
    "snapshot_features_rfm.csv",
    index=False
)

### Feature Group 2: Trend Features

#### Purpose

* Measure changes in customer purchasing behavior over time.
* Identify whether a customer is improving, remaining stable, or declining.

#### Method

* The 180-day observation window is divided into two equal 90-day periods.

**Period A (Previous 90 Days)**

* Snapshot Date − 180 days
* to
* Snapshot Date − 90 days

**Period B (Recent 90 Days)**

* Snapshot Date − 90 days
* to
* Snapshot Date

#### Features Created

**Spend Trend**

* Spend Trend = Recent Spend ÷ Previous Spend
* Measures change in customer spending.

**Frequency Trend**

* Frequency Trend = Recent Orders ÷ Previous Orders
* Measures change in purchase frequency.

**AOV Trend**

* AOV Trend = Recent AOV ÷ Previous AOV
* Measures change in average order value.

#### Interpretation

* Trend > 1 → Customer behavior is improving.
* Trend = 1 → Customer behavior is stable.
* Trend < 1 → Customer behavior is declining.

#### Importance

* Trend features capture behavioral changes that cannot be identified using RFM metrics alone.
* These features help detect early signs of customer value decline and improve predictive model performance.


#Part A - Feature 5: Spend Trend

In [ ]:
#Step 1 — Create Previous 90-Day Window

# ====================================================
# PREVIOUS 90 DAYS
# ====================================================

# Orders between:
# Snapshot -180 days
# and
# Snapshot -90 days

previous_90d = history_180d[
    (
        history_180d["InvoiceDate"]
        <
        (history_180d["snapshot_date"] - pd.Timedelta(days=90))
    )
].copy()

print("Previous 90 Days Shape:")
print(previous_90d.shape)

previous_90d.head()

Previous 90 Days Shape:
(50427, 5)


,Customer ID,snapshot_date,Invoice,InvoiceDate,Revenue
12,12490,2010-05-31,496325,2010-01-31 13:05:00,320.26
13,12490,2010-05-31,496916,2010-02-04 15:33:00,51.00
14,12490,2010-05-31,497377,2010-02-09 11:37:00,534.26
36,12682,2010-05-31,490959,2009-12-08 15:14:00,507.90
37,12682,2010-05-31,492944,2009-12-21 13:18:00,138.75


In [ ]:
#Step 2 — Create Recent 90-Day Window
# ====================================================
# RECENT 90 DAYS
# ====================================================

# Orders between:
# Snapshot -90 days
# and
# Snapshot

recent_90d = history_180d[
    (
        history_180d["InvoiceDate"]
        >=
        (history_180d["snapshot_date"] - pd.Timedelta(days=90))
    )
].copy()

print("Recent 90 Days Shape:")
print(recent_90d.shape)

recent_90d.head()

Recent 90 Days Shape:
(46550, 5)


,Customer ID,snapshot_date,Invoice,InvoiceDate,Revenue
15,12490,2010-05-31,500121,2010-03-04 14:05:00,202.35
16,12490,2010-05-31,500582,2010-03-09 09:46:00,400.20
17,12490,2010-05-31,507315,2010-05-07 14:56:00,435.16
33,12533,2010-05-31,501927,2010-03-22 10:56:00,437.97
42,12682,2010-05-31,500653,2010-03-09 11:10:00,689.45


In [ ]:
#Step 3 — Verify

print(previous_90d.shape)
print(recent_90d.shape)

(50427, 5)
(46550, 5)


Window Split Validation
Total Orders in 180-Day History = 96,977
Previous 90 Days = 50,427 orders
Recent 90 Days = 46,550 orders

Validation:

50,427 + 46,550 = 96,977

In [ ]:
#Step 4 — Previous Spend
# ====================================================
# PREVIOUS SPEND
# ====================================================

previous_spend = (
    previous_90d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Revenue"]
    .sum()
    .reset_index()
)

previous_spend.rename(
    columns={
        "Revenue": "previous_spend"
    },
    inplace=True
)

previous_spend.head()

,Customer ID,snapshot_date,previous_spend
0,12346,2010-06-30,117.05
1,12346,2010-07-31,27.05
2,12346,2010-09-30,142.31
3,12346,2010-10-31,142.31
4,12346,2010-11-30,142.31


In [ ]:
#Step 5 — Recent Spend
# ====================================================
# RECENT SPEND
# ====================================================

recent_spend = (
    recent_90d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Revenue"]
    .sum()
    .reset_index()
)

recent_spend.rename(
    columns={
        "Revenue": "recent_spend"
    },
    inplace=True
)

recent_spend.head()

,Customer ID,snapshot_date,recent_spend
0,12346,2010-06-30,142.31
1,12346,2010-07-31,142.31
2,12346,2010-08-31,142.31
3,12346,2011-01-31,77183.60
4,12346,2011-02-28,77183.60


In [ ]:
#Step 6 — Merge Into Snapshot Features
# ====================================================
# MERGE SPEND FEATURES
# ====================================================

snapshot_features = snapshot_features.merge(
    previous_spend,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

snapshot_features = snapshot_features.merge(
    recent_spend,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

snapshot_features.head()

,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d,previous_spend,recent_spend
0,12362,2010-05-31,180.0,0.0,0.00,0.000000,NaN,NaN
1,12490,2010-05-31,23.0,6.0,1943.23,323.871667,905.52,1037.71
2,12533,2010-05-31,69.0,1.0,437.97,437.970000,NaN,437.97
3,12636,2010-05-31,180.0,0.0,0.00,0.000000,NaN,NaN
4,12682,2010-05-31,24.0,9.0,5325.20,591.688889,3178.62,2146.58


In [ ]:
#Step 7 — Fill Missing Values
snapshot_features["previous_spend"] = (
    snapshot_features["previous_spend"]
    .fillna(0)
)

snapshot_features["recent_spend"] = (
    snapshot_features["recent_spend"]
    .fillna(0)
)

In [ ]:
#Step 8 — Create Spend Trend
#This is important because we need to avoid division by zero.
# ====================================================
# SPEND TREND
# ====================================================
import numpy as np
snapshot_features["spend_trend"] = np.where(
    snapshot_features["previous_spend"] > 0,
    snapshot_features["recent_spend"]
    /
    snapshot_features["previous_spend"],
    1
)

In [ ]:
#Step 9 — Audit Spend Trend
snapshot_features["spend_trend"].describe()

,spend_trend
count,46098.000000
mean,1.007724
std,4.409616
min,0.000000
25%,0.616163
50%,1.000000
75%,1.000000
max,430.224000


In [ ]:
# How many snapshots have zero spend
# in the previous 90 days?

print(
    (snapshot_features["previous_spend"] == 0).sum()
)

25058


In [ ]:
# How many snapshots have spend
# in the recent 90 days?

print(
    (snapshot_features["recent_spend"] > 0).sum()
)

19480


In [ ]:
# Most important:
# Previous spend = 0
# Recent spend > 0

print(
    (
        (snapshot_features["previous_spend"] == 0)
        &
        (snapshot_features["recent_spend"] > 0)
    ).sum()
)

6114


# Part B - Feature 6: Frequency Trend

In [ ]:
#Step 1 — Previous Orders
# ====================================================
# PREVIOUS ORDERS
# ====================================================

previous_orders = (
    previous_90d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Invoice"]
    .nunique()
    .reset_index()
)

previous_orders.rename(
    columns={
        "Invoice": "previous_orders"
    },
    inplace=True
)

In [ ]:
#Step 2 — Recent Orders
# ====================================================
# RECENT ORDERS
# ====================================================

recent_orders = (
    recent_90d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Invoice"]
    .nunique()
    .reset_index()
)

recent_orders.rename(
    columns={
        "Invoice": "recent_orders"
    },
    inplace=True
)

In [ ]:
#Step 3 — Merge
snapshot_features = snapshot_features.merge(
    previous_orders,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

snapshot_features = snapshot_features.merge(
    recent_orders,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

In [ ]:
#Step 4 — Fill Missing Values
snapshot_features["previous_orders"] = (
    snapshot_features["previous_orders"]
    .fillna(0)
)

snapshot_features["recent_orders"] = (
    snapshot_features["recent_orders"]
    .fillna(0)
)

In [ ]:
#Step 5 — Frequency Trend
import numpy as np

snapshot_features["frequency_trend"] = np.where(
    snapshot_features["previous_orders"] > 0,
    snapshot_features["recent_orders"]
    /
    snapshot_features["previous_orders"],
    1
)

In [ ]:
#Step 6 — Audit
snapshot_features["frequency_trend"].describe()

,frequency_trend
count,46098.000000
mean,0.891243
std,0.632441
min,0.000000
25%,0.666667
50%,1.000000
75%,1.000000
max,13.000000


# Part C - Feature 7: AOV Trend

Business Question
Is the customer spending more or less
per order recently?

Example:

Previous AOV = £200

Recent AOV = £100

AOV Trend = 0.50

Customer order size is shrinking.

In [ ]:
#Step 1 — Previous AOV
# ====================================================
# PREVIOUS AOV
# ====================================================

previous_aov = (
    previous_spend
    .merge(
        previous_orders,
        on=["Customer ID", "snapshot_date"],
        how="left"
    )
)

previous_aov["previous_aov"] = np.where(
    previous_aov["previous_orders"] > 0,
    previous_aov["previous_spend"]
    /
    previous_aov["previous_orders"],
    0
)

previous_aov = previous_aov[
    [
        "Customer ID",
        "snapshot_date",
        "previous_aov"
    ]
]

In [ ]:
#Step 2 — Recent AOV
# ====================================================
# RECENT AOV
# ====================================================

recent_aov = (
    recent_spend
    .merge(
        recent_orders,
        on=["Customer ID", "snapshot_date"],
        how="left"
    )
)

recent_aov["recent_aov"] = np.where(
    recent_aov["recent_orders"] > 0,
    recent_aov["recent_spend"]
    /
    recent_aov["recent_orders"],
    0
)

recent_aov = recent_aov[
    [
        "Customer ID",
        "snapshot_date",
        "recent_aov"
    ]
]

In [ ]:
#Step 3 — Merge
snapshot_features = snapshot_features.merge(
    previous_aov,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

snapshot_features = snapshot_features.merge(
    recent_aov,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

In [ ]:
#Step 4 — Fill Missing
snapshot_features["previous_aov"] = (
    snapshot_features["previous_aov"]
    .fillna(0)
)

snapshot_features["recent_aov"] = (
    snapshot_features["recent_aov"]
    .fillna(0)
)

In [ ]:
#Step 5 — AOV Trend
snapshot_features["aov_trend"] = np.where(
    snapshot_features["previous_aov"] > 0,
    snapshot_features["recent_aov"]
    /
    snapshot_features["previous_aov"],
    1
)

In [ ]:
#Step 6 — Audit
snapshot_features["aov_trend"].describe()

,aov_trend
count,46098.000000
mean,0.951959
std,5.461928
min,0.000000
25%,0.738428
50%,1.000000
75%,1.000000
max,964.677966


### Feature Group 3: Lifetime Features


#Part A - Feature 8: CUSTOMER AGE

**Business Question**

* How long has the customer been active?

**Formula**

* Customer Age = Snapshot Date − First Purchase Date

**Interpretation**

* Higher Age → More established customer
* Lower Age → Newer customer


In [ ]:
# ====================================================
# FEATURE 8 : CUSTOMER AGE
# ====================================================

# First purchase date for each customer

first_purchase = (
    invoice_level
    .groupby("Customer ID")["InvoiceDate"]
    .min()
    .reset_index()
)

first_purchase.rename(
    columns={
        "InvoiceDate": "first_purchase_date"
    },
    inplace=True
)

# Merge

snapshot_features = snapshot_features.merge(
    first_purchase,
    on="Customer ID",
    how="left"
)

# Customer age in days

snapshot_features["customer_age_days"] = (
    snapshot_features["snapshot_date"]
    -
    snapshot_features["first_purchase_date"]
).dt.days

snapshot_features[
    [
        "Customer ID",
        "snapshot_date",
        "customer_age_days"
    ]
].head()

,Customer ID,snapshot_date,customer_age_days
0,12362,2010-05-31,180
1,12490,2010-05-31,180
2,12533,2010-05-31,180
3,12636,2010-05-31,180
4,12682,2010-05-31,180


In [ ]:
snapshot_features["customer_age_days"].describe()

,customer_age_days
count,46098.000000
mean,362.442449
std,120.956687
min,180.000000
25%,260.000000
50%,346.000000
75%,452.000000
max,637.000000


#Part B - Feature 9: LIFETIME ORDERS

In [ ]:
# ====================================================
# FEATURE 9 : LIFETIME ORDERS
# ====================================================

# Count all orders before each snapshot

lifetime_orders = (
    history_df
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Invoice"]
    .nunique()
    .reset_index()
)

lifetime_orders.rename(
    columns={
        "Invoice": "lifetime_orders"
    },
    inplace=True
)

# Merge feature

snapshot_features = snapshot_features.merge(
    lifetime_orders,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

# Fill missing values

snapshot_features["lifetime_orders"] = (
    snapshot_features["lifetime_orders"]
    .fillna(0)
)

snapshot_features["lifetime_orders"].describe()

,lifetime_orders
count,46098.000000
mean,5.804265
std,10.336530
min,1.000000
25%,2.000000
50%,3.000000
75%,6.000000
max,303.000000


#Part C - Feature 10: LIFETIME SPEND

In [ ]:
# ====================================================
# FEATURE 10 : LIFETIME SPEND
# ====================================================

# Sum all revenue before each snapshot

lifetime_spend = (
    history_df
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Revenue"]
    .sum()
    .reset_index()
)

lifetime_spend.rename(
    columns={
        "Revenue": "lifetime_spend"
    },
    inplace=True
)

# Merge feature

snapshot_features = snapshot_features.merge(
    lifetime_spend,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

# Fill missing values

snapshot_features["lifetime_spend"] = (
    snapshot_features["lifetime_spend"]
    .fillna(0)
)

# Audit

snapshot_features["lifetime_spend"].describe()

,lifetime_spend
count,46098.000000
mean,2784.905104
std,11587.166487
min,2.950000
25%,380.970000
50%,961.100000
75%,2322.590000
max,479701.580000


What We Have Built

Core RFM Features
1. recency_days
2. frequency_180d
3. monetary_180d
4. aov_180d

Trend Features
5. spend_trend
6. frequency_trend
7. aov_trend

Lifetime Features
8. customer_age_days
9. lifetime_orders
10. lifetime_spend

In [ ]:
# ====================================================
# SAVE FEATURE DATASET
# ====================================================

snapshot_features.to_csv(
    "snapshot_features_v1.csv",
    index=False
)

print(snapshot_features.shape)

(46098, 19)


In [ ]:
# ====================================================
# SAVE FEATURE DATASET
# ====================================================

snapshot_features.to_pickle(
    "/content/drive/MyDrive/Project files/snapshot_features.pkl"
)

print("snapshot_features saved successfully")

snapshot_features saved successfully
